In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau 
import pandas as pd
import numpy as np
import os
import cv2 
from tqdm import tqdm 
import time 
import random

random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: ---{DEVICE}----- uwu")

MODEL_LOAD_PATH = '/kaggle/input/hypertenstion-pretraining/output_models/pretrain_weights.pth'
MODEL_SAVE_PATH = 'output_models/hypertensive_fine_tuned_best_model.pth'
TRAIN_CSV = '/kaggle/input/hypertension-data-cleaning-2-0/output_processed_metadata/hr_train_split.csv'
VAL_CSV = '/kaggle/input/hypertension-data-cleaning-2-0/output_processed_metadata/hr_val_split.csv'
os.makedirs('output_models', exist_ok=True)

Using: ---cuda----- uwu


# Data Args

In [3]:
def aggressive_augs(image):

    # horizontal and vert flips
    if random.random() > 0.5:
        image = cv2.flip(image, 1) # hor
    if random.random() > 0.5:
        image = cv2.flip(image, 0) # vert

    rows, cols, _ = image.shape
    # 0-20 deg rotat. 10% scale dif
    M = cv2.getRotationMatrix2D((cols / 2, rows / 2), random.uniform(-20, 20), random.uniform(0.9, 1.1))
    
    # random shifts 0-10 pix
    tx = random.randint(-10, 10)
    ty = random.randint(-10, 10)
    M[0, 2] += tx
    M[1, 2] += ty
    
    image = cv2.warpAffine(image, M, (cols, rows), borderMode=cv2.BORDER_REFLECT)

    # brigtness and contress
    alpha = random.uniform(0.8, 1.2)  # og 1
    beta = random.uniform(-10, 10) #og 0
    image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    # color jitter
    
    hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    # hue
    h_shift = random.randint(-10, 10)
    hsv_image[:, :, 0] = cv2.add(hsv_image[:, :, 0], h_shift) 
    
    # saturation
    s_shift = random.randint(-20, 20)
    hsv_image[:, :, 1] = cv2.add(hsv_image[:, :, 1], s_shift)
    

    image = cv2.cvtColor(hsv_image, cv2.COLOR_HSV2RGB)
    return image

In [4]:
TRANSFORMS_TENSOR = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# DATASET

In [5]:
class hyptertesive_dataset(Dataset):

    def __init__(self, csv_file, is_train=False):
        self.df = pd.read_csv(csv_file)
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['full_image_path']
        image = cv2.imread(img_path)

        if image is None:
            print(f"==========fail to load img=============")

        image = cv2.resize(image, (224, 224))
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 


        if self.is_train:
            image = aggressive_augs(image)

        image = TRANSFORMS_TENSOR(image) 
        # float
        label = float(self.df.iloc[idx]['label_binary'])
        
        return image, label


In [6]:
BATCH_SIZE = 16


train_dataset = hyptertesive_dataset(TRAIN_CSV, is_train=True)
val_dataset = hyptertesive_dataset(VAL_CSV, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4) 
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4) 

print('LOADERRS HAS COMPLET')

LOADERRS HAS COMPLET


In [7]:
model = models.densenet201(pretrained=False)

model.classifier = nn.Linear(model.classifier.in_features, 1)

model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=DEVICE), strict=False)
print('model load good')

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


model load good


In [8]:
for param in model.parameters():
        param.requires_grad = True

model.to(DEVICE)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [9]:
LR_TOP = 1e-3
LR_DEEP = 1e-5


optimizer = optim.AdamW([
    {'params': model.classifier.parameters(), 'lr': LR_TOP},
    {'params': model.features.parameters(), 'lr': LR_DEEP}
])

POS_CLASS_WEIGHT = 1.0

pos_weights = torch.tensor(POS_CLASS_WEIGHT, dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

In [10]:
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=10,
    verbose=True
)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


# TRAINING!!!!

In [11]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in tqdm(dataloader, desc=f"training"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad() 
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Grad clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def val_epoch(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc=f"Validating"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
    return total_loss / len(dataloader)

In [12]:
NUM_EPOCH = 150
EPOCH_PAITENCE = 41

best_loss = float('inf')
epochs_no_improve = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCH+1):
    epoch_start_time = time.time()
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = val_epoch(model, val_loader, criterion)
    
    scheduler.step(val_loss)

    epoch_time = time.time() - epoch_start_time

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        epochs_no_improve = 0
        print(f'     ----model has SAVED and finally improved at {epoch+1} Best loss is {val_loss:.4f}')
    else:
        epochs_no_improve += 1
        print(f'     ----model has stopped improving from the best loss of {best_loss:.4f}.   ({epochs_no_improve}/{EPOCH_PAITENCE})')

        if epochs_no_improve >= EPOCH_PAITENCE:
            print('=======Early stopping has trigger=======')
            print('Training stopped after {epochs_no_improve} epoches without improvement ')
            break

print(f"\nFINETUNNING complete in JSUTUUTTUSUTUTTT!! [{(time.time() - start_time) / 60:.2f}] minutes.")

print("FINETUNINGINGINGN ISISISISISISSS finished!!!! YIPEPEPEPEEE")
print(f"=== FINAL best weights are are : {MODEL_SAVE_PATH}) ==== AND best loss is: {best_loss:.4f} === ")

Validating: 100%|██████████| 7/7 [00:01<00:00,  4.18it/s]


     ----model has SAVED and finally improved at 2 Best loss is 0.5687


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.40it/s]


     ----model has SAVED and finally improved at 3 Best loss is 0.5317


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.65it/s]


...

     ----model has stopped improving from the best loss of 0.3104.   (39/41)


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.59it/s]


     ----model has stopped improving from the best loss of 0.3104.   (40/41)


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.84it/s]

     ----model has stopped improving from the best loss of 0.3104.   (41/41)
=======Early stopping has trigger=======
Training stopped after {epochs_no_improve} epoches without improvement 

FINETUNNING complete in JSUTUUTTUSUTUTTT!! [15.90] minutes.
FINETUNINGINGINGN ISISISISISISSS finished!!!! YIPEPEPEPEEE
=== FINAL best weights are are : output_models/hypertensive_fine_tuned_best_model.pth) ==== AND best loss is: 0.3104 === 
